# Notebook 3 — Feature Engineering

Generates RoughPy log-signature, GARCH, and combined features for every selected experiment. Processing is fit on training data only; validation and test sets use the training medians and scalers.

In [1]:
import warnings
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from arch import arch_model
import roughpy as rp

c:\Users\kyler\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_DIR = Path("../data/datasets")
FEATURE_DIR = Path("../data/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [4]:
SIGNATURE_DEPTH = 2
GARCH_FEATURE_NAMES = [
    "garch_mu", "garch_omega", "garch_alpha", "garch_beta", "garch_persistence",
    "conditional_volatility_last", "conditional_volatility_mean", "conditional_volatility_std",
    "conditional_volatility_min", "conditional_volatility_max",
    "standardized_residual_last", "absolute_residual_last",
]

def add_time_channel(sequence):
    sequence = np.asarray(sequence, dtype=np.float64)
    time = np.linspace(0.0, 1.0, sequence.shape[0]).reshape(-1, 1)
    return np.concatenate([time, sequence], axis=1)

def roughpy_logsignature(sequence, depth=2):
    path = add_time_channel(sequence)
    increments = np.diff(path, axis=0)

    width = increments.shape[1]

    ctx = rp.get_context(
        width=width,
        depth=depth,
        coeffs=rp.DPReal
    )

    stream = rp.LieIncrementStream.from_increments(
        increments,
        ctx=ctx
    )

    interval = rp.RealInterval(
        0.0,
        float(len(increments))
    )

    log_signature = stream.log_signature(
        interval
    )

    return np.asarray(
        log_signature,
        dtype=np.float64
    ).reshape(-1)

def create_logsignature_features(X, depth=2, description="Signatures"):
    return np.vstack([roughpy_logsignature(seq, depth) for seq in tqdm(X, desc=description)])

In [5]:
def extract_garch_features(sequence):
    returns_pct = np.asarray(sequence[:, 0], dtype=np.float64) * 100.0
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result = arch_model(
                returns_pct, mean="Constant", vol="GARCH", p=1, q=1,
                dist="normal", rescale=False
            ).fit(disp="off", show_warning=False)
        params = result.params
        vol = np.asarray(result.conditional_volatility, dtype=np.float64)
        resid = np.asarray(result.resid, dtype=np.float64)
        std_resid = resid / np.where(vol == 0, np.nan, vol)
        alpha = params.get("alpha[1]", np.nan)
        beta = params.get("beta[1]", np.nan)
        return np.array([
            params.get("mu", np.nan), params.get("omega", np.nan), alpha, beta, alpha + beta,
            vol[-1], vol.mean(), vol.std(), vol.min(), vol.max(), std_resid[-1], abs(resid[-1])
        ], dtype=np.float64)
    except Exception:
        return np.full(len(GARCH_FEATURE_NAMES), np.nan, dtype=np.float64)

def create_garch_features(X, description="GARCH"):
    return np.vstack([extract_garch_features(seq) for seq in tqdm(X, desc=description)])

def fit_medians(X):
    X = np.asarray(X, dtype=np.float64).copy()
    X[~np.isfinite(X)] = np.nan
    medians = np.nanmedian(X, axis=0)
    return np.where(np.isnan(medians), 0.0, medians)

def apply_medians(X, medians):
    X = np.asarray(X, dtype=np.float64).copy()
    X[~np.isfinite(X)] = np.nan
    rows, cols = np.where(np.isnan(X))
    X[rows, cols] = medians[cols]
    return X

In [6]:
for experiment in RUN_EXPERIMENTS:
    print()
    print("=" * 80)
    print(experiment)
    print("=" * 80)

    input_dir = DATASET_DIR / experiment
    output_dir = FEATURE_DIR / experiment
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train = np.load(input_dir / "X_train.npy")
    X_validation = np.load(input_dir / "X_validation.npy")
    X_test = np.load(input_dir / "X_test.npy")

    y_train = np.load(input_dir / "y_train.npy")
    y_validation = np.load(input_dir / "y_validation.npy")
    y_test = np.load(input_dir / "y_test.npy")

    sig_train = create_logsignature_features(
        X_train,
        SIGNATURE_DEPTH,
        f"{experiment} signature train"
    )

    sig_val = create_logsignature_features(
        X_validation,
        SIGNATURE_DEPTH,
        f"{experiment} signature validation"
    )

    sig_test = create_logsignature_features(
        X_test,
        SIGNATURE_DEPTH,
        f"{experiment} signature test"
    )

    garch_train = create_garch_features(
        X_train,
        f"{experiment} GARCH train"
    )

    garch_val = create_garch_features(
        X_validation,
        f"{experiment} GARCH validation"
    )

    garch_test = create_garch_features(
        X_test,
        f"{experiment} GARCH test"
    )

    sig_medians = fit_medians(sig_train)
    garch_medians = fit_medians(garch_train)

    sig_train = apply_medians(sig_train, sig_medians)
    sig_val = apply_medians(sig_val, sig_medians)
    sig_test = apply_medians(sig_test, sig_medians)

    garch_train = apply_medians(garch_train, garch_medians)
    garch_val = apply_medians(garch_val, garch_medians)
    garch_test = apply_medians(garch_test, garch_medians)

    sig_scaler = StandardScaler().fit(sig_train)
    garch_scaler = StandardScaler().fit(garch_train)

    sig_train = sig_scaler.transform(sig_train)
    sig_val = sig_scaler.transform(sig_val)
    sig_test = sig_scaler.transform(sig_test)

    garch_train = garch_scaler.transform(garch_train)
    garch_val = garch_scaler.transform(garch_val)
    garch_test = garch_scaler.transform(garch_test)

    combined_train = np.concatenate(
        [sig_train, garch_train],
        axis=1
    )

    combined_val = np.concatenate(
        [sig_val, garch_val],
        axis=1
    )

    combined_test = np.concatenate(
        [sig_test, garch_test],
        axis=1
    )

    arrays = {
        "X_signature_train": sig_train,
        "X_signature_validation": sig_val,
        "X_signature_test": sig_test,

        "X_garch_train": garch_train,
        "X_garch_validation": garch_val,
        "X_garch_test": garch_test,

        "X_combined_train": combined_train,
        "X_combined_validation": combined_val,
        "X_combined_test": combined_test,

        "y_train": y_train,
        "y_validation": y_validation,
        "y_test": y_test,
    }

    for name, arr in arrays.items():
        np.save(
            output_dir / f"{name}.npy",
            arr
        )

    for split in ["train", "validation", "test"]:
        metadata = pd.read_csv(
            input_dir / f"meta_{split}.csv"
        )

        metadata.to_csv(
            output_dir / f"meta_{split}.csv",
            index=False
        )

    joblib.dump(
        {
            "signature_depth": SIGNATURE_DEPTH,
            "signature_medians": sig_medians,
            "garch_medians": garch_medians,
            "signature_scaler": sig_scaler,
            "garch_scaler": garch_scaler,
            "garch_feature_names": GARCH_FEATURE_NAMES,
        },
        output_dir / "preprocessing.joblib"
    )

    print(
        "Saved:",
        experiment,
        "| Signature:",
        sig_train.shape,
        "| GARCH:",
        garch_train.shape,
        "| Combined:",
        combined_train.shape
    )


daily_ff5


daily_ff5 GARCH test: 100%|██████████| 59150/59150 [06:59<00:00, 141.13it/s]


Saved: daily_ff5 | Signature: (275975, 36) | GARCH: (275975, 12) | Combined: (275975, 48)
